# IT135IU Group 3, Topic 8: D2 Data and profiling
**Decision question:** which film genres should the Content Marketing and Promotion Team promote to each gender x age-group segment? **Data:** MovieLens 1M (GroupLens).

| No. | Role | Name | Student ID |
|-----|------|------|------------|
| 1 | Product owner | Truc Phuong | ITDSIU25031 |
| 2 | Data engineer | Le Quan | ITDSIU25033 |
| 3 | Data analyst A | Que | ITDSIU25036 |
| 4 | Data analyst B | Hoang Quan | ITDSIU25035 |
| 5 | Visualisation lead | Tuyen | ITDSIU23028 |
| 6 | Reproducibility lead | Trung | ITDSIU24054 |
| 7 | Ethics and privacy officer | Cuong | ITDSIU25008 |

## Source and terms of use
- **Dataset:** MovieLens 1M, https://grouplens.org/datasets/movielens/1m/ (file loaded in Step 1 from https://files.grouplens.org/datasets/movielens/ml-1m.zip)
- **Publisher:** GroupLens Research, University of Minnesota
- **Date of retrieval:**
- **Terms of use:**


### Step 1: Load the Three Raw Tables
Load the raw files (`ratings.dat`, `users.dat`, and `movies.dat`) from the MovieLens zip file.

In [ ]:
import io
import urllib.request
import zipfile
from typing import Any
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
READ: dict[str, Any] = {
    "sep": "::",
    "engine": "python",
    "encoding": "latin-1",
    "header": None,
}

AGE_LABEL = {
    1: "Under 18",
    18: "18-24",
    25: "25-34",
    35: "35-44",
    45: "45-49",
    50: "50-55",
    56: "56+",
}

def load(archive: zipfile.ZipFile, name: str, cols: list[str]) -> pd.DataFrame:
    """Read one "::" separated member straight out of the archive."""
    data = io.BytesIO(archive.read(f"ml-1m/{name}"))
    return pd.read_csv(data, names=cols, **READ)

print("STEP 1  load the three raw tables")
zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
print(f"members : {zf.namelist()}")

ratings = load(zf, "ratings.dat", ["user_id", "movie_id", "rating", "timestamp"])
users = load(zf, "users.dat", ["user_id", "gender", "age", "occupation", "zip"])
movies = load(zf, "movies.dat", ["movie_id", "title", "genres"])

STEP 1  load the three raw tables
members : ['ml-1m/', 'ml-1m/movies.dat', 'ml-1m/ratings.dat', 'ml-1m/README', 'ml-1m/users.dat']


### Step 2: Missing Values, Keys, and Orphan Rows
Validate the integrity of our tables by checking for missing values, confirming primary keys are unique, and identifying any orphaned foreign keys.

In [ ]:
print("STEP 2  missing values, keys and orphan rows")

for name, table in [
    ("ratings", ratings),
    ("users", users),
    ("movies", movies),
]:
    print(f"\n{name}: shape={table.shape}")
    print(table.isna().sum().to_string())

print("\nprimary key uniqueness")
print(f"  users.user_id   : {users['user_id'].is_unique}")
print(f"  movies.movie_id : {movies['movie_id'].is_unique}")

bad_user = ~ratings["user_id"].isin(users["user_id"])
bad_movie = ~ratings["movie_id"].isin(movies["movie_id"])

n_bad_user = bad_user.sum()
n_bad_movie = bad_movie.sum()
d_bad_user = ratings.loc[bad_user, "user_id"].nunique()
d_bad_movie = ratings.loc[bad_movie, "movie_id"].nunique()

print("\norphan foreign keys in ratings")
print(f"  rows, unknown user_id     : {n_bad_user:,}")
print(f"  rows, unknown movie_id    : {n_bad_movie:,}")
print(f"  distinct unknown user_id  : {d_bad_user}")
print(f"  distinct unknown movie_id : {d_bad_movie}")

STEP 2  missing values, keys and orphan rows

ratings: shape=(1000209, 4)
user_id      0
movie_id     0
rating       0
timestamp    0

users: shape=(6040, 5)
user_id       0
gender        0
age           0
occupation    0
zip           0

movies: shape=(3883, 3)
movie_id    0
title       0
genres      0

primary key uniqueness
  users.user_id   : True
  movies.movie_id : True

orphan foreign keys in ratings
  rows, unknown user_id     : 0
  rows, unknown movie_id    : 0
  distinct unknown user_id  : 0
  distinct unknown movie_id : 0


### Step 2b: Data Profile of the Three Raw Tables
Rows, columns, types, missing values, distinct values and ranges for each raw table (the profile table required by the brief).

In [ ]:
print("STEP 2b  data profile of the raw tables")

def profile(table: pd.DataFrame, name: str) -> pd.DataFrame:
    """One row per column: dtype, missing, distinct values and numeric range."""
    rows = []
    for col in table.columns:
        s = table[col]
        numeric = pd.api.types.is_numeric_dtype(s)
        rows.append(
            {
                "column": col,
                "dtype": str(s.dtype),
                "missing": int(s.isna().sum()),
                "unique": int(s.nunique()),
                "min": s.min() if numeric else "-",
                "max": s.max() if numeric else "-",
            }
        )
    print(f"\n{name}: {table.shape[0]:,} rows x {table.shape[1]} columns")
    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
    return out

for name, table in [("ratings", ratings), ("users", users), ("movies", movies)]:
    profile(table, name)

STEP 2b  data profile of the raw tables

ratings: 1,000,209 rows x 4 columns
   column dtype  missing  unique       min        max
  user_id int64        0    6040         1       6040
 movie_id int64        0    3706         1       3952
   rating int64        0       5         1          5
timestamp int64        0  458455 956703932 1046454590

users: 6,040 rows x 5 columns
    column  dtype  missing  unique min  max
   user_id  int64        0    6040   1 6040
    gender object        0       2   -    -
       age  int64        0       7   1   56
occupation  int64        0      21   0   20
       zip object        0    3439   -    -

movies: 3,883 rows x 3 columns
  column  dtype  missing  unique min  max
movie_id  int64        0    3883   1 3952
   title object        0    3883   -    -
  genres object        0     301   -    -


### Step 2c: Duplicate, Validity and Coverage Checks
Duplicate rating pairs, valid value ranges, rating period, and films that no one rated.

In [ ]:
print("STEP 2c  duplicate, validity and coverage checks")

n_dup_pairs = ratings.duplicated(subset=["user_id", "movie_id"]).sum()
n_dup_rows = ratings.duplicated().sum()
print("duplicate rows in ratings")
print(f"  repeated (user_id, movie_id) pairs : {n_dup_pairs:,}")
print(f"  fully identical rows               : {n_dup_rows:,}")

print("\nvalid value ranges")
print(f"  rating values : {[int(x) for x in sorted(ratings['rating'].unique())]}")
print(f"  gender codes  : {sorted(users['gender'].unique())}")
stamps = pd.to_datetime(ratings["timestamp"], unit="s", utc=True)
print(f"  first rating  : {stamps.min()}")
print(f"  last rating   : {stamps.max()}")

rated_ids = set(ratings["movie_id"])
n_unrated = (~movies["movie_id"].isin(rated_ids)).sum()
print("\nfilm coverage")
print(f"  films in movies.dat      : {len(movies):,}")
print(f"  films with >= 1 rating   : {len(rated_ids):,}")
print(f"  films never rated        : {n_unrated:,}")
print(f"  highest movie_id         : {movies['movie_id'].max():,}")
print(f"  unused ids up to highest : {movies['movie_id'].max() - len(movies):,}")

STEP 2c  duplicate, validity and coverage checks
duplicate rows in ratings
  repeated (user_id, movie_id) pairs : 0
  fully identical rows               : 0

valid value ranges
  rating values : [1, 2, 3, 4, 5]
  gender codes  : ['F', 'M']
  first rating  : 2000-04-25 23:05:32+00:00
  last rating   : 2003-02-28 17:49:50+00:00

film coverage
  films in movies.dat      : 3,883
  films with >= 1 rating   : 3,706
  films never rated        : 177
  highest movie_id         : 3,952
  unused ids up to highest : 69


### Step 3: Merge Into One Working Table
Merge the `ratings`, `users`, and `movies` DataFrames using left joins.

In [ ]:
print("STEP 3  merge into one working table")

df = ratings.merge(users, on="user_id", how="left", validate="m:1")
df = df.merge(movies, on="movie_id", how="left", validate="m:1")

assert len(df) == len(ratings), "left merge changed the row count"

print(f"shape     : {df.shape}")
print(f"columns   : {df.columns.tolist()}")
print(f"total NaN : {df.isna().sum().sum()}")

STEP 3  merge into one working table
shape     : (1000209, 10)
columns   : ['user_id', 'movie_id', 'rating', 'timestamp', 'gender', 'age', 'occupation', 'zip', 'title', 'genres']
total NaN : 0


### Step 3b: Drop Personal Columns
`zip` and `occupation` are dropped from the working table: zip code combined with gender and age band could narrow down an individual, and neither column is used in the analysis (as committed in D1).

In [ ]:
print("STEP 3b  drop personal columns")

before = df.shape[1]
df = df.drop(columns=["zip", "occupation"])
print(f"columns before : {before}")
print(f"columns after  : {df.shape[1]}")
print(f"columns        : {df.columns.tolist()}")
assert len(df) == len(ratings), "dropping columns changed the row count"

STEP 3b  drop personal columns
columns before : 10
columns after  : 8
columns        : ['user_id', 'movie_id', 'rating', 'timestamp', 'gender', 'age', 'title', 'genres']


### Step 4: Derived Columns (Year and Age Group)
Extract the release year from movie titles and map numeric age codes to readable categories.

In [ ]:
print("STEP 4  derived columns, year and age_group")

df["year"] = (
    df["title"].str.extract(r"\((\d{4})\)\s*$", expand=False).astype("Int64")
)

print(f"rows with NaN year : {df['year'].isna().sum()}")
print(f"year.min()         : {df['year'].min()}")
print(f"year.max()         : {df['year'].max()}")

codes = df["age"]
df["age_group"] = codes.map(AGE_LABEL)

assert df["age_group"].isna().sum() == 0, "unmapped age code found"

print(f"\nrows with NaN age_group : {df['age_group'].isna().sum()}")

age_order = [AGE_LABEL[code] for code in sorted(AGE_LABEL)]
print(df["age_group"].value_counts().reindex(age_order).to_string())

STEP 4  derived columns, year and age_group
rows with NaN year : 0
year.min()         : 1919
year.max()         : 2000

rows with NaN age_group : 0
age_group
Under 18     27211
18-24       183536
25-34       395556
35-44       199003
45-49        83633
50-55        72490
56+          38780


In [ ]:
print("STEP 4b  release-year coverage")

years_rated = set(df["year"].dropna().astype(int))
years_all = set(
    movies["title"].str.extract(r"\((\d{4})\)\s*$", expand=False).dropna().astype(int)
)
span = range(min(years_all), max(years_all) + 1)
print(f"distinct years, rated films in df : {len(years_rated)}")
print(f"distinct years, all of movies.dat : {len(years_all)}")
print(f"years in the span with no film    : {[y for y in span if y not in years_all]}")

STEP 4b  release-year coverage
distinct years, rated films in df : 81
distinct years, all of movies.dat : 81
years in the span with no film    : [1924]


### Step 5: Genre Breakdown
Explode the concatenated genres so each row represents a single rating-genre pair.

In [ ]:
print("STEP 5  genre breakdown")

genres = df.copy()
genres["genre_list"] = genres["genres"].str.split("|")
genres["n_genres"] = genres["genre_list"].str.len()

print("n_genres per rating row")
print(genres["n_genres"].describe().to_string())

exploded = genres.explode("genre_list").rename(columns={"genre_list": "genre"})

print(f"\nrows after explode : {len(exploded):,}")
print(f"ratio to ratings   : {len(exploded) / len(ratings):.4f}")
print(f"distinct genres    : {exploded['genre'].nunique()}")
print(exploded["genre"].value_counts().to_string())

STEP 5  genre breakdown
n_genres per rating row
count    1.000209e+06
mean     2.101376e+00
std      9.768456e-01
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      6.000000e+00

rows after explode : 2,101,815
ratio to ratings   : 2.1014
distinct genres    : 18
genre
Comedy         356580
Drama          354529
Action         257457
Thriller       189680
Sci-Fi         157294
Romance        147523
Adventure      133953
Crime           79541
Horror          76386
Children's      72186
War             68527
Animation       43293
Musical         41533
Mystery         40178
Fantasy         36301
Western         20683
Film-Noir       18261
Documentary      7910


In [ ]:
print("STEP 5b  labels per film and genre combinations")

per_film_all = movies["genres"].str.count(r"\|") + 1
per_film_rated = per_film_all[movies["movie_id"].isin(rated_ids)]
print(f"distinct genre combinations         : {movies['genres'].nunique()}")
print(f"mean labels per rating row (df)     : {genres['n_genres'].mean():.4f}")
print(f"mean labels per film, all movies    : {per_film_all.mean():.4f}")
print(f"mean labels per film, rated movies  : {per_film_rated.mean():.4f}")

STEP 5b  labels per film and genre combinations
distinct genre combinations         : 301
mean labels per rating row (df)     : 2.1014
mean labels per film, all movies    : 1.6503
mean labels per film, rated movies  : 1.6708


### Step 6: Segment Sparsity
Identify low-density segments (gender $\times$ age_group $\times$ genre) with thin data support.

In [ ]:
print("STEP 6  segment sparsity")

cells = (
    exploded.groupby(["gender", "age_group", "genre"], observed=True)
    .agg(n_ratings=("rating", "size"), n_users=("user_id", "nunique"))
    .reset_index()
    .sort_values("n_users")
)

cells["age_group"] = pd.Categorical(
    cells["age_group"], age_order, ordered=True
)

print(f"cells present    : {len(cells)}")
print(f"full grid 2x7x18 : {2 * 7 * 18}")
print("\nn_users per cell")
print(cells["n_users"].describe().round(1).to_string())

for limit in (30, 50, 100):
    thin = cells[cells["n_users"] < limit]
    share = len(thin) / len(cells)
    print(
        f"\nunder {limit} users: {len(thin)} of {len(cells)} cells ({share:.1%})"
    )
    rows = zip(
        thin["gender"],
        thin["age_group"],
        thin["genre"],
        thin["n_users"],
        thin["n_ratings"],
    )
    for gender, age, genre, n_users, n_ratings in rows:
        label = f"{gender} {age}"
        print(
            f"  {label:<12} {genre:<12} {n_users:>3} users, {n_ratings:>5} ratings"
        )

STEP 6  segment sparsity
cells present    : 252
full grid 2x7x18 : 252

n_users per cell
count     252.0
mean      372.6
std       354.7
min        11.0
25%       133.5
50%       270.5
75%       460.5
max      1538.0

under 30 users: 3 of 252 cells (1.2%)
  F Under 18   Documentary   11 users,    25 ratings
  F Under 18   Film-Noir     26 users,    80 ratings
  F 56+        Documentary   29 users,    72 ratings

under 50 users: 5 of 252 cells (2.0%)
  F Under 18   Documentary   11 users,    25 ratings
  F Under 18   Film-Noir     26 users,    80 ratings
  F 56+        Documentary   29 users,    72 ratings
  F Under 18   Western       35 users,    91 ratings
  M Under 18   Documentary   45 users,   105 ratings

under 100 users: 41 of 252 cells (16.3%)
  F Under 18   Documentary   11 users,    25 ratings
  F Under 18   Film-Noir     26 users,    80 ratings
  F 56+        Documentary   29 users,    72 ratings
  F Under 18   Western       35 users,    91 ratings
  M Under 18   Documentary 

### Step 7: Freeze the Thresholds into a Tier Column
Categorize segments into quality tiers (`exclude`, `low_confidence`, `ok`) based on user count thresholds.

In [ ]:
print("STEP 7  freeze the thresholds into a tier column")

TIER_EDGES = [0, 30, 100, np.inf]
TIER_NAMES = ["exclude", "low_confidence", "ok"]

cells["tier"] = pd.cut(
    cells["n_users"], bins=TIER_EDGES, labels=TIER_NAMES, right=False
)

tier_counts = cells["tier"].value_counts().reindex(TIER_NAMES)

assert tier_counts.sum() == len(cells) == 2 * 7 * 18, "tiers miss some cells"

print("cells per tier")
print(tier_counts.to_string())

low = cells[cells["tier"] == "low_confidence"]
print(f"\nlow_confidence cells by segment ({len(low)} total)")
print(low.groupby(["gender", "age_group"], observed=True).size().to_string())

STEP 7  freeze the thresholds into a tier column
cells per tier
tier
exclude             3
low_confidence     38
ok                211

low_confidence cells by segment (38 total)
gender  age_group
F       Under 18     16
        18-24         1
        45-49         1
        50-55         3
        56+          13
M       Under 18      3
        56+           1


### Step 7b: Filter Out Cells Below the Minimum Size
Data manipulation operation: filtering. Cells with fewer than 30 distinct users are excluded; the thresholds were frozen in Step 7 before any satisfaction rate was computed.

In [ ]:
print("STEP 7b  filter out cells below the minimum size")

kept = cells[cells["tier"] != "exclude"]
dropped = cells[cells["tier"] == "exclude"]

print(f"cells before filter : {len(cells)}")
print(f"cells after filter  : {len(kept)}")
print(f"cells excluded      : {len(dropped)}")
for gender, age, genre, n_users in zip(
    dropped["gender"], dropped["age_group"], dropped["genre"], dropped["n_users"]
):
    print(f"  {gender} {age:<9} {genre:<12} {n_users:>3} users")

STEP 7b  filter out cells below the minimum size
cells before filter : 252
cells after filter  : 249
cells excluded      : 3
  F Under 18  Documentary   11 users
  F Under 18  Film-Noir     26 users
  F 56+       Documentary   29 users


### Step 8: Satisfaction Rate with Wilson Intervals
Calculate 95% Wilson score confidence intervals for user satisfaction rates (ratings $\ge$ 4) across segments.

In [ ]:
print("STEP 8  satisfaction rate with Wilson intervals")

Z = 1.959963984540054  # two sided 95 percent normal quantile

def wilson(count, nobs, z=Z):
    """Wilson lower and upper bound for count successes out of nobs trials."""
    p = count / nobs
    denom = 1 + z**2 / nobs
    centre = (p + z**2 / (2 * nobs)) / denom
    halfwidth = z / denom * np.sqrt(p * (1 - p) / nobs + z**2 / (4 * nobs**2))
    return centre - halfwidth, centre + halfwidth

exploded["satisfied"] = exploded["rating"] >= 4

rates = (
    exploded.groupby(["gender", "age_group", "genre"], observed=True)
    .agg(
        n_ratings=("rating", "size"),
        n_users=("user_id", "nunique"),
        n_satisfied=("satisfied", "sum"),
        sat_rate=("satisfied", "mean"),
    )
    .reset_index()
)

rates = rates.merge(
    cells[["gender", "age_group", "genre", "tier"]],
    on=["gender", "age_group", "genre"],
    how="left",
    validate="1:1",
)

rates["ci_low"], rates["ci_high"] = wilson(
    rates["n_satisfied"], rates["n_ratings"]
)

in_unit = rates["sat_rate"].between(0, 1).all()
ci_brackets = (rates["ci_low"] <= rates["sat_rate"]).all() and (
    rates["sat_rate"] <= rates["ci_high"]
).all()

assert in_unit, "sat_rate outside [0, 1]"
assert ci_brackets, "sat_rate falls outside its own interval"

print(f"cells                      : {len(rates)}")
print(f"sat_rate within [0, 1]     : {in_unit}")
print(f"ci_low <= sat_rate <= high : {ci_brackets}")

print(f"overall sat_rate, no explode: {(df['rating'] >= 4).mean():.4f}")
print(f"sat_rate across cells, min  : {rates['sat_rate'].min():.4f}")
print(f"sat_rate across cells, max  : {rates['sat_rate'].max():.4f}")

rates["ci_width"] = rates["ci_high"] - rates["ci_low"]
print("\nmean interval width by tier")
for tier in TIER_NAMES:
    width = rates.loc[rates["tier"] == tier, "ci_width"].mean()
    print(f"  {tier:<15} {width:.4f}")

STEP 8  satisfaction rate with Wilson intervals
cells                      : 252
sat_rate within [0, 1]     : True
ci_low <= sat_rate <= high : True
overall sat_rate, no explode: 0.5752
sat_rate across cells, min  : 0.3835
sat_rate across cells, max  : 0.8104

mean interval width by tier
  exclude         0.2449
  low_confidence  0.0880
  ok              0.0332


### Step 9: Data Profile of the Working Table
Rows, columns, types, missing values, distinct values and ranges for the merged table `df` (after dropping personal columns and adding `year` and `age_group`).

In [ ]:
print("STEP 9  data profile of the working table")

_ = profile(df, "df (working table)")

STEP 9  data profile of the working table

df (working table): 1,000,209 rows x 10 columns
   column  dtype  missing  unique       min        max
  user_id  int64        0    6040         1       6040
 movie_id  int64        0    3706         1       3952
   rating  int64        0       5         1          5
timestamp  int64        0  458455 956703932 1046454590
   gender object        0       2         -          -
      age  int64        0       7         1         56
    title object        0    3706         -          -
   genres object        0     301         -          -
     year  Int64        0      81      1919       2000
age_group object        0       7         -          -


## Data manipulation operations (index)
| Operation | Where | What it does |
|-----------|-------|--------------|
| Merging two tables | Step 3 | left joins ratings, users and movies with validate="m:1" and asserts the row count |
| Creating a derived column | Steps 4, 7, 8 | year and age_group (Step 4), tier (Step 7), satisfied flag (Step 8) |
| Filtering | Step 7b | removes cells with fewer than 30 distinct users, counts printed before and after |
| Grouping and aggregation | Steps 6, 8 | users, ratings and satisfaction rate per gender x age_group x genre cell |

Step 5 (explode genres into one row per rating-genre pair) prepares the data for Steps 6 and 8 and is not counted as a fifth operation.


## Inputs for the other roles
Environment and column lists printed from this notebook, so teammates take them from here instead of typing them by hand.

In [ ]:
print("HANDOFF  environment and column lists")

import platform
from importlib.metadata import version

print(f"python  : {platform.python_version()}")
for pkg in ("numpy", "pandas"):
    print(f"{pkg:<8}: {version(pkg)}")
print(f"seed    : {SEED}")

print("\nworking table df, columns")
for col, dtype in df.dtypes.items():
    print(f"  {col:<12} {dtype}")

print("\ncell table rates, columns")
for col, dtype in rates.dtypes.items():
    print(f"  {col:<12} {dtype}")

HANDOFF  environment and column lists
python  : 3.13.16
numpy   : 2.1.3
pandas  : 2.2.3
seed    : 42

working table df, columns
  user_id      int64
  movie_id     int64
  rating       int64
  timestamp    int64
  gender       object
  age          int64
  title        object
  genres       object
  year         Int64
  age_group    object

cell table rates, columns
  gender       object
  age_group    object
  genre        object
  n_ratings    int64
  n_users      int64
  n_satisfied  int64
  sat_rate     float64
  tier         category
  ci_low       float64
  ci_high      float64
  ci_width     float64


## Cleaning log

Every figure below is printed by the cell named in the "Step" column of the run on 2026-10-07. Row counts refer to the 1,000,209-row ratings table unless stated otherwise.

| ID | Step | Check | Finding | Decision | Reason | Before → After |
|----|------|-------|---------|----------|--------|----------------|
| C1 | 2 | Missing values in `ratings`, `users`, `movies` | 0 missing in every column of all three tables | Keep as is | Nothing to impute or drop | 1,000,209 / 6,040 / 3,883 rows → unchanged |
| C2 | 2 | Primary key uniqueness | `users.user_id` and `movies.movie_id` are unique | Keep as is | Joins are many-to-one, so no row can multiply | unchanged |
| C3 | 2 | Orphan foreign keys in `ratings` | 0 rows with an unknown `user_id`, 0 with an unknown `movie_id` | Keep as is | Every rating can be joined | unchanged |
| C4 | 2c | Duplicate ratings | 0 repeated (`user_id`, `movie_id`) pairs, 0 fully identical rows | Nothing removed | No duplicates exist | 1,000,209 → 1,000,209 |
| C5 | 2c | Valid value ranges | `rating` ∈ {1,2,3,4,5}; `gender` ∈ {F, M}; ratings dated 2000-04-25 to 2003-02-28 | Keep as is | All values are inside the documented domain | unchanged |
| C6 | 2c | Films nobody rated | 3,883 films in `movies.dat`, 3,706 with at least 1 rating, 177 never rated | Left joins start from `ratings`, so the 177 films do not appear in the working table | A film with no rating carries no preference signal | 3,883 → 3,706 films in `df` |
| C7 | 3 | Merge `ratings` + `users` + `movies` | Left joins with `validate="m:1"`; row count asserted equal to `ratings` | Merged into one working table | One row per rating with user and film attributes | 1,000,209 → 1,000,209 rows; 4 → 10 columns |
| C8 | 3b | Personal columns | `zip` (3,439 distinct) could single out a person when combined with gender and age band; `occupation` is not used in the analysis | Dropped `zip` and `occupation` | Privacy and relevance, as committed in D1 | 10 → 8 columns |
| C9 | 4 / 4b | Release year parsed from `title` | 0 unparsed titles; years 1919–2000; 81 distinct years in rated films and in `movies.dat`; the only year without a film inside the span is 1924 | Added `year` | Needed for any release-period cut | 8 → 9 columns |
| C10 | 4 | Age code to label | 0 unmapped codes; 7 groups: Under 18 (27,211), 18-24 (183,536), 25-34 (395,556), 35-44 (199,003), 45-49 (83,633), 50-55 (72,490), 56+ (38,780); counts sum to 1,000,209 | Added `age_group` | Readable and ordered labels for the age segments | 9 → 10 columns |
| C11 | 5 | Multi-genre films | A rating row carries 1–6 genre labels (mean 2.1014); 301 distinct genre strings; 18 distinct genres | Exploded to one row per rating-genre pair, in a separate table `exploded`; `df` is untouched | The unit of analysis is the genre | 1,000,209 → 2,101,815 rows (×2.1014) |
| C12 | 6 / 7 | Sparsity of the 252 segments (2 genders × 7 age groups × 18 genres) | Distinct users per segment: min 11, median 270.5, max 1,538. Under 30 users: 3 segments; under 50: 5; under 100: 41 | Froze tiers before any satisfaction rate was computed: `exclude` < 30, `low_confidence` 30–99, `ok` ≥ 100 users | Thresholds fixed in advance, so they cannot be tuned to the outcome | 252 segments → 3 / 38 / 211 |
| C13 | 7b | Filter segments below the minimum size | Excluded: F Under 18 Documentary (11 users), F Under 18 Film-Noir (26), F 56+ Documentary (29) | Removed the 3 `exclude` segments from the segment table | Too few users for a stable rate (mean interval width 0.2449 vs 0.0332 for `ok`) | 252 → 249 segments |
| C14 | 8 | Satisfaction rate and uncertainty | `satisfied` = rating ≥ 4; 95% Wilson interval; all rates in [0, 1] and inside their own interval | Kept; interval width reported with every rate | A rate without its uncertainty would over-rank thin segments. Mean width by tier: `low_confidence` 0.0880, `ok` 0.0332 | 252 segment rows with rate and interval |
| C15 | 9 | Profile of the final working table `df` | 1,000,209 rows × 10 columns, 0 missing, 81 distinct years, 7 age groups | Final | Table handed to the analysts | 10 columns |

### Decisions still open
- Number of tiers: two (exclude, ok) or three (adds `low_confidence`). Currently three, with the cut-offs 30 and 100.
- Whether to filter by user activity: not done, because the notebook does not print a ratings-per-user profile. Add one if the team wants it.

### Things that do NOT change the row count
- Steps 5 and 6 never modify `df`; they build `exploded` and the segment table from it. The segment counts in C12 and C13 are counted in distinct users, so a user who rated several genres appears in several segments. Segment sizes therefore do not add up to 6,040.

## PDF export
Only after **Runtime > Restart session and run all** has finished without error and the notebook is saved (Ctrl+S) with its outputs. Quickest route: File > Print > Save as PDF (check that tables do not overflow the page).

In [ ]:
import json
import subprocess
import sys

EXPORT = True  # set to False to skip
BASENAME = "D2_Topic08_LeQuan"


def run(*args):
    subprocess.run([sys.executable, "-m", *args], check=True)


if EXPORT:
    from google.colab import _message, files

    live = _message.blocking_request(
        "get_ipynb", request="", timeout_sec=120
    )["ipynb"]
    for cell in live["cells"]:
        cell["metadata"].pop("executionInfo", None)

    with open(f"{BASENAME}.ipynb", "w", encoding="utf-8") as f:
        json.dump(live, f, ensure_ascii=False, indent=1)

    run("pip", "install", "-q", "nbconvert[webpdf]")
    run("playwright", "install", "--with-deps", "chromium")
    run("jupyter", "nbconvert", "--to", "webpdf", f"{BASENAME}.ipynb")

    files.download(f"{BASENAME}.ipynb")
    files.download(f"{BASENAME}.pdf")
else:
    print("export skipped (EXPORT is False)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>